In [ ]:
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
from shapely.geometry import LineString

def get_road_network():
    while True:
        place_state = input("State in the USA: ")
        place_city = input("City in given state: ")
        place_name = f"{place_city}, {place_state}, USA"
        
        try:
            # attempt to get road network
            graph = ox.graph_from_place(place_name, network_type="drive")

            # check if graph empty
            if graph.number_of_nodes() == 0:
                raise ValueError(f"No data found for {place_name}. Please try again.")
            
            print(f"Successfully fetched road network for {place_name}.")
            return graph

        except ValueError as e:
            print(f"Error: {e}")
        except Exception as e:
            print(f"Unexpected error: {e}")
            break

# get road network
graph = get_road_network()

# copy of graph
G = graph.copy()

# Get source and target nodes
def get_coordinates(graph, label):
    while True:
        try:
            lat = float(input(f"Enter {label} latitude: "))
            lon = float(input(f"Enter {label} longitude: "))

            node = ox.nearest_nodes(graph, X=lon, Y=lat)
            print(f"{label} Node: {node}")
            return node

        except ValueError:
            print("Invalid latitude or longitude. Please enter valid numbers.")
        except Exception as e:
            print(f"Error finding nearest node: {e}. Try again.")

source_node = get_coordinates(graph, "Source")
target_node = get_coordinates(graph, "Target")

if not nx.has_path(G, source_node, target_node):
    print("No path exists between source and target. Exiting.")
    exit()

# Dijkstra path calculation (shortest path by distance)
shortest_path = nx.shortest_path(
    G, source=source_node, target=target_node, weight='length'
)

path_length_meters = sum(
    graph[u][v][k]['length'] for u, v, k in zip(shortest_path[:-1], shortest_path[1:])
)
path_length_miles = path_length_meters / 1609.34

print(f"Length (miles): {path_length_miles}")

fig, ax = ox.plot_graph(graph, node_size=0, bgcolor='white', show=False, close=False)

for u, v in zip(shortest_path[:-1], shortest_path[1:]):
    edge = graph[u][v][0]
    if 'geometry' in edge:
        line = edge['geometry']
    else:
        line = LineString([(graph.nodes[u]['x'], graph.nodes[u]['y']),
                           (graph.nodes[v]['x'], graph.nodes[v]['y'])])
    ax.plot(*line.xy, linewidth=4, color='blue')
plt.show()


KeyboardInterrupt: Interrupted by user